In [1]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')
import workflow_objects as wfo
import load_tools as ldt
import logging_config as lcf
import database_interact as dbi

logger = lcf.get_logger('workflow_example')

pd.options.display.max_colwidth = 300

## 1. Setup Configuration and File Paths

First, we load the configuration and define the compound input files.

In [2]:
# Load configuration
WORKFLOW_CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
ATLAS_CONFIG = ldt.load_atlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/create_atlas_config.yaml")

# Define compound input files - these must have columns 'inchi_key' and 'label'
COMPOUND_INPUTS = [
    "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv",
    "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv", 
    "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv",
    "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv",
    "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"
]

# Atlas input files - these need additional columns for atlas creation
ATLAS_INPUTS = [
    "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv",
    "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"
]

PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
PROJECT_FILES_PATH = f"{WORKFLOW_CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"

# Control flags
new_main_database = False
new_atlases = False
new_analysis = True

## 2. Create Main Database and Load Compounds

Using the new `DatabaseManager` class, we can create the main database and load compounds in a single step.

In [3]:
if new_main_database:
    # Create DatabaseManager instance
    db_manager = wfo.DatabaseManager(WORKFLOW_CONFIG)
    
    # Complete database setup: create database and load all compounds
    db_manager.setup_database_with_compounds(
        compound_file_paths=COMPOUND_INPUTS,
        overwrite=True,  # Set to True to overwrite existing database
        retrieve_pubchem=True  # Set to True to fetch PubChem metadata
    )
    
    logger.info("Main database created and compounds loaded successfully!")
else:
    logger.info("Skipping main database creation (new_main_database=False)")

2025-09-09 16:16:51 | metatlas2.workflow_example | INFO | Skipping main database creation (new_main_database=False)


## 3. Create Atlases from Database

Now we can create atlases using the `AtlasManager` class.

In [5]:
if new_atlases:
    # Create AtlasManager instance
    atlas_manager = wfo.AtlasManager(ATLAS_CONFIG)
    
    # Create multiple atlases from configurations
    created_atlases = atlas_manager.create_multiple_atlases()
    
    # Display created atlases
    logger.info("Atlases created successfully!")
    atlas_uids = {}
    for atlas_uid, atlas_data in created_atlases.items():
        logger.info(f"  - {atlas_data['type']}")
        logger.info(f"    - {atlas_data['chromatography']}")
        logger.info(f"      - {atlas_data['polarity']}")
        logger.info(f"        - {atlas_data['name']}: {atlas_uid}")
        atlas_uids[atlas_data['type']] = {}
        atlas_uids[atlas_data['type']][f"{atlas_data['chromatography']}_{atlas_data['polarity']}"] = atlas_uid

else:
    atlas_uids = {}
    atlas_uids['qc'] = {}
    atlas_uids['istd'] = {}
    atlas_uids['ema'] = {}
    atlas_uids['qc']['hilicz_positive'] = "atl-raw-qc-c540ce5104764bd08ed80037e6a0e664"
    atlas_uids['istd']['hilicz_positive'] = "atl-raw-istd-dabb01aced3243fcbdfe963e5bfd7a5c"
    atlas_uids['ema']['hilicz_positive'] = "atl-raw-ema-48647fd583d44c13aca9f1d5d72f0636"
    logger.info("Using existing atlas UIDs")

2025-09-09 16:17:23 | metatlas2.workflow_example | INFO | Using existing atlas UIDs


In [6]:
if new_analysis:

    # Setup project paths
    PROJECT_DIRECTORY = f"{WORKFLOW_CONFIG['paths']['projects_dir']}/{PROJECT_NAME}"
    PROJECT_DB_PATH = f"{PROJECT_DIRECTORY}/{PROJECT_NAME}.duckdb"

    # Ensure project directory exists
    Path(PROJECT_DIRECTORY).mkdir(parents=True, exist_ok=True)

    workflow = wfo.TargetedMetabolomicsWorkflow(
        config=WORKFLOW_CONFIG,
        project_db_path=PROJECT_DB_PATH,
        project_directory=PROJECT_DIRECTORY,
        main_db_path=WORKFLOW_CONFIG["paths"]["main_database"],
        atlas_uids=atlas_uids
    )

    result = workflow.run_complete_workflow(stop_at_stage=wfo.WorkflowStage.MANUAL_CURATION)

    if result:
        logger.info("Workflow reached manual curation stage")
        logger.info("GUI is ready for manual compound annotation")
        logger.info("After manual curation, run: workflow.continue_to_final_report()")
    else:
        logger.info("Workflow completed successfully")

2025-09-09 16:17:28 | metatlas2.workflow_objects | INFO | === Stage 1: Project Setup ===
2025-09-09 16:17:28 | metatlas2.database_interact | INFO | Retrieved atlas metadata for: HILICZ QC Atlas Positive
2025-09-09 16:17:28 | metatlas2.workflow_objects | INFO | Validated atlas atl-raw-qc-c540ce5104764bd08ed80037e6a0e664 for qc_hilicz_positive
2025-09-09 16:17:28 | metatlas2.database_interact | INFO | Retrieved atlas metadata for: HILICZ ISTD Atlas Positive
2025-09-09 16:17:28 | metatlas2.workflow_objects | INFO | Validated atlas atl-raw-istd-dabb01aced3243fcbdfe963e5bfd7a5c for istd_hilicz_positive
2025-09-09 16:17:28 | metatlas2.database_interact | INFO | Retrieved atlas metadata for: HILICZ EMA Atlas Positive
2025-09-09 16:17:28 | metatlas2.workflow_objects | INFO | Validated atlas atl-raw-ema-48647fd583d44c13aca9f1d5d72f0636 for ema_hilicz_positive
2025-09-09 16:17:28 | metatlas2.workflow_objects | INFO | Atlas coverage: {'hilicz_positive': {'qc_atlas': 'atl-raw-qc-c540ce5104764bd08e

In [ ]:
def complete_workflow_example(config: dict, project_name: str):
    """
    Complete example showing upstream atlas creation + targeted analysis workflow.
    This demonstrates the full pipeline from CSV files to final analysis results.
    """
    
    # Setup project paths
    project_directory = f"{config['paths']['projects_dir']}/{project_name}"
    project_db_path = f"{project_directory}/{project_name}.duckdb"
    
    # Ensure project directory exists
    Path(project_directory).mkdir(parents=True, exist_ok=True)
    
    # Initialize targeted metabolomics workflow
    workflow = wfo.TargetedMetabolomicsWorkflow(
        config=config,
        project_db_path=project_db_path,
        project_directory=project_directory,
        main_db_path=config["paths"]["main_database"],
        atlas_uids=atlas_uids
    )
    
    # Run complete workflow
    try:
        logger.info("Starting targeted analysis workflow...")
        result = workflow.run_complete_workflow(stop_at_stage=wfo.WorkflowStage.MANUAL_CURATION)
        
        if result:  # GUI returned for manual curation
            logger.info("Workflow reached manual curation stage")
            logger.info("GUI is ready for manual compound annotation")
            logger.info("After manual curation, run: workflow.continue_to_final_report()")
            return result, workflow
        else:
            logger.info("Workflow completed successfully")
            return None, workflow
            
    except Exception as e:
        logger.error(f"Workflow failed: {e}")
        raise
    
    return workflow